# Few-shot prompt design for Intent-to-SLO (Qwen3.5 4B / 9B)

**Status: design only.** No model has been run with these prompts yet; the experiment policy (which set-ups, in what order, how many runs) is still to be defined. This notebook records (1) what is wrong or missing in the paper's few-shot prompt, (2) the six curated examples and why each was chosen, (3) the two prompt set-ups that were built from them, and (4) the checks that the prompts are well-formed.

**Decisions taken by the researcher**

| item | decision |
|---|---|
| Models | Qwen3.5-4B and 9B only (27B skipped: the goal is lightweight models) |
| Set-up 1, `fs1_verbatim+3_prompt.txt` | the paper's few-shot prompt as published (system prompt + its 3 visible examples verbatim) **plus 3 carefully chosen examples** in place of the 3 undisclosed ones |
| Set-up 2, `fs2_unitFixed+6_prompt.txt` | all unit issues fixed in the prompt; the paper's 3 examples **replaced by 3 new examples**; the other 3 are the ones added in set-up 1; the examples together should be representative of the dataset; a **precautionary message** makes the model verify the snapshot for highest/lowest requests |
| Hypothesis for set-up 2 | the explicit verification rule will force the model to read the snapshot (fewer `value: null` answers on highest/lowest requests) |
| Files | `prompts/fs1_verbatim+3_prompt.txt`, `prompts/fs2_unitFixed+6_prompt.txt` (each = system prompt + user template, separated by `=== SYSTEM ===` / `=== USER ===`); generator `src/fewshot.py`; tests `tests/test_fewshot.py` |

**Assumptions I made (please confirm or overrule)**

1. **Extremum scope = the requested layer.** "On the edge, pick the node with the lowest CPU utilisation" is answered over edge nodes only (what the real records do). 61 synthetic gold SLOs (about 5%) use the global extremum instead and cannot match this convention.
2. **Paper's three header lines omitted.** The paper's system prompt begins with `Service: {{service}} / Intent Expression: {{expression}} / Snapshot (JSON): {{snapshot}}`; as in all our zero-shot runs these lines are left out, so the only difference between zero-shot and fs1 is the examples.
3. **Format as in the paper:** all examples sit in the single user message ("Example n / Input / Output"), not as chat turns.
4. **Invented services and snapshots.** Examples use `svc/example-*` services and small snapshots in the real schema (trimmed to the grounding fields), so nothing is copied from the 716 records.
5. **The added unit-neutral/unit-carrying split.** Examples 4-6 (shared by both set-ups) contain no unit conversion, so they do not contradict the paper's own prompt in set-up 1; the unit conventions are carried by examples 1-3, which appear only in set-up 2.
6. **Three clarifications inside the snapshot rule of set-up 2** go slightly beyond "verify the snapshot": layer scope, skipping null nodes, and the `>=`/`<=` mapping for highest/lowest. They are listed in the change log and can be removed as a separate ablation.

In [1]:
import difflib, json, re, subprocess
from collections import Counter
from pathlib import Path
import sys
ROOT = Path.cwd()
assert (ROOT / "src").is_dir(), "start the notebook from the llmExp/ folder"
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import fewshot as F
from src import analysis as A
from src.data_io import load_records, load_snapshot, snapshot_compact
from src.parse import parse_response
from src.prompts import build_messages, load_fs_prompt, build_messages_fs
from src.score import canon_list, load_predictions

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 140)
records = load_records()
recs = [r for r in records if r["meta"]["status"] == "valid"]
snapshot = load_snapshot()

def sig_of(slos):  # P placement, E explicit threshold, I implicit (highest/lowest)
    return "".join(sorted("P" if t[0] == "placement_location" else ("I" if t[3] in ("highest", "lowest") else "E") for t in canon_list(slos)))
def key_of(t): return (t[0], t[1], t[2], t[3])                      # type, metric, comparison, extremum tag
def kind_of(t):
    if t[0] == "placement_location": return f"placement={t[4]}"
    return f"{t[1]} {t[3]}" if t[3] in ("highest", "lowest") else f"{t[1]} {t[2]} (explicit)"
def norm(expr, service): return re.sub(r"\d+(\.\d+)?", "N", expr.replace(service, "<SVC>")).strip().rstrip(".?")

paper_slos = {k: parse_response(v.split("Output:\n", 1)[1])["parsed"]["slos"] for k, v in F.PAPER_EXAMPLES.items()}
ex_slos = {k: e["slos"] for k, e in F.EXAMPLES.items()}
gold = [canon_list(r["label"]["slos"]) for r in recs]
print(len(recs), "valid records,", sum(len(g) for g in gold), "gold SLOs")

521 valid records, 1169 gold SLOs


## 1. The paper's few-shot prompt: what it contains and what is missing

Source: Fig. 9 (page 16) of *Intent Engine* (Islam & Calheiros, J. Syst. Archit. 179 (2026) 103938), read from the PDF. The system prompt is the zero-shot system prompt; the user message holds 6 hard-coded examples followed by the query. **Only 3 examples are printed** ("3 more examples...."). The three visible examples are reproduced verbatim below (braces resolved: the paper's template is a Python f-string).

In [2]:
for k, text in F.PAPER_EXAMPLES.items():
    print(f"----- paper example {k[1]} -----"); print(text); print()

----- paper example 1 -----
Input:
{
  "service": "svc/example-auth",
  "expression": "Deploy svc/example-auth, keep CPU utilization below 55 percent.",
  "snapshot": {"nodes": [{"name": "n1", "cpu_utilization": 40}, {"name": "n2", "cpu_utilization": 62}]}
}
Output:
{
  "service": "svc/example-auth",
  "slos": [
    {
      "metric": "cpu_utilization",
      "target": {"cmp": "<=", "op": null, "value": 55, "range": null},
      "type": "dynamic_compute_resource",
      "unit": "percent"
    }
  ]
}

----- paper example 2 -----
Input:
{
  "service": "svc/example-cache",
  "expression": "Run svc/example-cache with at least 2 GB memory.",
  "snapshot": {"nodes": [{"name": "n1", "memory_capacity": 1073741824}, {"name": "n2", "memory_capacity": 8589934592}]}
}
Output:
{
  "service": "svc/example-cache",
  "slos": [
    {
      "metric": "memory_capacity",
      "target": {"cmp": ">=", "op": null, "value": 2000000000, "range": null},
      "type": "static_compute_resource",
      "unit": "by

In [3]:
# Evidence for each comment on the paper's examples, computed from the data
vis = {key_of(canon_list(s)[0]) for s in paper_slos.values()}              # kinds shown by the 3 visible examples
kinds = Counter(kind_of(t) for g in gold for t in g)
tot = sum(kinds.values())
share = lambda pred: sum(1 for g in gold for t in g if pred(t)) / tot
ev = {
 "gold SLOs that are placement": share(lambda t: t[0] == "placement_location"),
 "gold SLOs that are highest/lowest lookups": share(lambda t: t[3] in ("highest", "lowest")),
 "gold SLOs of a kind shown by the 3 visible examples (type, metric, cmp, no extremum)": share(lambda t: key_of(t) in vis),
 "records with 3 or more SLOs": sum(len(g) >= 3 for g in gold) / len(gold),
}
display(pd.Series(ev).map("{:.1%}".format).rename("share").to_frame())

units = Counter()
for r in recs:
    e = r["input"]["intent"]["expression"]
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(GB|GiB|MB|MiB|KB|KiB|TB|TiB|CPU cores|millicores)\b", e): units[m.group(2)] += 1
print("units written in the 521 intents:", dict(units))
conv = Counter()
for r in recs:
    e = r["input"]["intent"]["expression"]
    for t in canon_list(r["label"]["slos"]):
        m = re.search(r"(\d+(?:\.\d+)?)\s*GB", e)
        if t[1] in ("memory_capacity", "storage_capacity") and t[3] is None and m and isinstance(t[4], float):
            conv["gold = N x 2^30 (binary)" if abs(t[4] - float(m.group(1)) * 2**30) < 1 else ("gold = N x 10^9 (decimal)" if abs(t[4] - float(m.group(1)) * 1e9) < 1 else "other")] += 1
print("gold convention for 'N GB' requests:", dict(conv))
print("paper example 2 teaches: 2 GB ->", paper_slos["P2"][0]["target"]["value"], "(decimal);  binary would be", 2 * 2**30)
cpu = [r for r in recs if any(t[1] == "cpu_capacity" and t[3] is None for t in canon_list(r["label"]["slos"]))]
print("explicit cpu_capacity requests:", len(cpu), "| all written as '<N> CPU cores':", all("CPU cores" in r["input"]["intent"]["expression"] for r in cpu),
      "| gold unit:", {s["unit"] for r in cpu for s in r["label"]["slos"] if s["metric"] == "cpu_capacity"}, "| paper prompt asks for: cores")

print("\nSnapshot field names: paper examples vs the real snapshot")
display(pd.DataFrame([
    {"metric": "cpu_utilization", "paper example snapshot": "cpu_utilization", "real snapshot": "utilization_now.cpu_util_percent"},
    {"metric": "memory_capacity", "paper example snapshot": "memory_capacity", "real snapshot": "memory_capacity_bytes"},
    {"metric": "storage_capacity", "paper example snapshot": "storage_capacity", "real snapshot": "utilization_now.storage_total_bytes"},
    {"metric": "layer (edge/cloud)", "paper example snapshot": "(absent)", "real snapshot": "layer"},
]).set_index("metric"))
print("fields per real node:", len(snapshot["nodes"][0]), "| the real nodes have", len(snapshot["nodes"]), "entries, 1 with all utilisation fields null")

,share
gold SLOs that are placement,33.4%
gold SLOs that are highest/lowest lookups,18.9%
"gold SLOs of a kind shown by the 3 visible examples (type, metric, cmp, no extremum)",24.5%
records with 3 or more SLOs,42.4%


units written in the 521 intents: {'GB': 158, 'CPU cores': 74}
gold convention for 'N GB' requests: {'gold = N x 2^30 (binary)': 154, 'other': 4}
paper example 2 teaches: 2 GB -> 2000000000 (decimal);  binary would be 2147483648
explicit cpu_capacity requests: 74 | all written as '<N> CPU cores': True | gold unit: {'millicores'} | paper prompt asks for: cores

Snapshot field names: paper examples vs the real snapshot


,paper example snapshot,real snapshot
metric,,
cpu_utilization,cpu_utilization,utilization_now.cpu_util_percent
memory_capacity,memory_capacity,memory_capacity_bytes
storage_capacity,storage_capacity,utilization_now.storage_total_bytes
layer (edge/cloud),(absent),layer


fields per real node: 16 | the real nodes have 6 entries, 1 with all utilisation fields null


**Finding.** The paper's three visible examples are all single explicit thresholds (CPU utilisation <= 55%, memory >= 2 GB, storage >= 500 MiB). Only **24.5%** of the gold SLOs are of a kind they show; none shows a placement SLO (33.4% of all gold SLOs) or a highest/lowest lookup (18.9%), and none has more than one SLO although 42.4% of records have three or more. **Unit conventions contradict the dataset:** Example 2 maps 2 GB to 2,000,000,000 bytes (decimal), whereas the gold labels use N x 2^30 in 154 of the 158 GB requests; Example 3 uses MiB, a unit that never occurs (the 521 intents contain only "GB" 158 times and "CPU cores" 74 times); no example touches CPU cores, which the intents state in cores and the gold stores as millicores (the paper's prompt asks for cores). **The example snapshots do not match the real one:** they use flat field names (`cpu_utilization`, `memory_capacity`, `storage_capacity`) and no `layer`, whereas the real nodes use `utilization_now.cpu_util_percent`, `memory_capacity_bytes`, `utilization_now.storage_total_bytes` and a `layer`; the snapshot is irrelevant to every visible example, so they never show how to read it. They also add an unused `"range": null` key and no example of rejecting an intent. Together this suggests the paper's few-shot gain comes mainly from showing the output format, not from helping with the hard cases.

## 2. Dataset profile that drives the choice of examples

In [4]:
sig = Counter(sig_of(r["label"]["slos"]) for r in recs)
sigdf = pd.DataFrame({"records": pd.Series(sig)}).sort_values("records", ascending=False)
sigdf["share"] = sigdf.records / len(recs)
sigdf["meaning"] = [" + ".join({"P": "placement", "E": "explicit", "I": "implicit"}[c] for c in s) for s in sigdf.index]
print("Record signatures (P placement, E explicit threshold/capacity, I implicit highest/lowest); top 10:")
display(sigdf.head(10).round(3))

kd = pd.Series(kinds).sort_values(ascending=False).to_frame("gold SLOs"); kd["share"] = kd["gold SLOs"] / tot
print("Gold SLO kinds:"); display(kd.round(3))

imp = Counter((t[1], t[3]) for g in gold for t in g if t[3] in ("highest", "lowest"))
impdf = pd.DataFrame([{"metric": m, "lowest": imp.get((m, "lowest"), 0), "highest": imp.get((m, "highest"), 0)} for m in sorted({m for m, _ in imp})]).set_index("metric")
impdf["total"] = impdf.sum(axis=1); impdf["share of implicit"] = impdf.total / impdf.total.sum()
print("Implicit (highest/lowest) SLOs by metric and direction:"); display(impdf.sort_values("total", ascending=False).round(3))

lay = Counter()
for r, g in zip(recs, gold):
    layer = next((t[4] for t in g if t[0] == "placement_location"), "none")
    for t in g:
        if t[3] in ("highest", "lowest"): lay[(r["meta"]["source"], layer)] += 1
print("Implicit SLOs by source x requested layer:", dict(lay))
print("SLO count per record:", dict(sorted(Counter(len(g) for g in gold).items())), "| cmp: utilisation thresholds are always <=, capacities always >=")

Record signatures (P placement, E explicit threshold/capacity, I implicit highest/lowest); top 10:


,records,share,meaning
EIP,90,0.173,explicit + implicit + placement
E,82,0.157,explicit
EP,80,0.154,explicit + placement
EEP,75,0.144,explicit + explicit + placement
P,55,0.106,placement
IP,41,0.079,implicit + placement
EE,36,0.069,explicit + explicit
EEIP,27,0.052,explicit + explicit + implicit + placement
EIIP,11,0.021,explicit + implicit + implicit + placement
IIP,6,0.012,implicit + implicit + placement


Gold SLO kinds:


,gold SLOs,share
placement=edge,196,0.168
placement=cloud,194,0.166
cpu_utilization <= (explicit),128,0.109
memory_utilization <= (explicit),100,0.086
storage_utilization <= (explicit),98,0.084
storage_capacity >= (explicit),83,0.071
memory_capacity >= (explicit),75,0.064
cpu_capacity >= (explicit),74,0.063
memory_capacity highest,30,0.026
cpu_utilization lowest,29,0.025


Implicit (highest/lowest) SLOs by metric and direction:


,lowest,highest,total,share of implicit
metric,,,,
memory_capacity,23,30,53,0.240
storage_utilization,23,24,47,0.213
cpu_utilization,29,10,39,0.176
port_utilization,29,6,35,0.158
storage_capacity,5,18,23,0.104
memory_utilization,14,4,18,0.081
cpu_capacity,5,1,6,0.027


Implicit SLOs by source x requested layer: {('real', 'edge'): 34, ('real', 'cloud'): 34, ('real', 'none'): 14, ('synthetic', 'cloud'): 79, ('synthetic', 'edge'): 60}
SLO count per record: {1: 137, 2: 163, 3: 178, 4: 43} | cmp: utilisation thresholds are always <=, capacities always >=


**Finding.** Records fall into a few output signatures: EIP 17.3%, E 15.7%, EP 15.4%, EEP 14.4%, P 10.6%, IP 7.9% (the top six cover 81% of records). Gold SLOs are 33.4% placement (edge 16.8%, cloud 16.6%, never both/any), 47.7% explicit (utilisation thresholds always `<=`: CPU 10.9%, memory 8.6%, storage 8.4%; capacities always `>=`: storage 7.1%, memory 6.4%, CPU 6.3%) and 18.9% implicit. Among the 221 lookups, memory capacity is the most frequent metric (24%), then storage utilisation (21%), CPU utilisation (18%), port utilisation (16%), storage capacity (10%), memory utilisation (8%) and CPU capacity (3%); 58% are `lowest`. **Almost all lookups name a layer** (68 of 82 in real records, all 139 in synthetic ones; only 14 have none), so the layer-restricted extremum is the norm and a model must apply it. Records contain 1-4 SLOs, with 3 the most common (178 records). These proportions are what the six examples are matched against.

## 3. How the six examples were chosen

**Principles**

1. **Represent output kinds, not just topics.** An example is chosen for the *signature* it shows (placement / explicit / implicit combination), the metrics and directions it covers, and the conventions it has to teach, in proportion to the dataset (section 2) but with the hard part (highest/lowest lookups) deliberately over-weighted.
2. **Teach what the models get wrong.** The 4B adds a spurious `highest` tag on plain thresholds (so a plain threshold with `op: null` must be shown); the 9B and 27B return `value: null` for most lookups and sometimes write the wrong operator (so lookups with the right `cmp`/`op` mapping must be shown).
3. **Every lookup example contains a trap that only correct reasoning avoids:** a node in the *other layer* has a lower/higher value, and/or a node has *null readings* that must be skipped. A model that ignores the layer or the nulls gets the example wrong.
4. **Outputs are computed, not typed:** each demonstrated value is calculated from the example snapshot by code, so labels follow the gold conventions exactly (layer-restricted extremum, nulls skipped, `<=` for lowest, `>=` for highest, binary data sizes, cores to millicores).
5. **Unit-neutral vs unit-carrying split.** Examples 4-6 contain no unit conversion, so they can be added to the paper's prompt (set-up 1) without contradicting its `cores` convention; the unit lessons are in examples 1-3, used only in set-up 2.
6. **No leakage and realistic wording.** Invented `svc/example-*` services; sentence frames differ from every dataset template, but the vocabulary is the dataset's ("CPU cores", "GB of memory", "below N%", "the node with the lowest current CPU utilization"). Because the dataset has a closed grammar (525 templates), part of any few-shot gain may reflect template familiarity; this is a caveat for the interpretation, not something the examples can remove.
7. **Realistic snapshots.** Real field names and nesting (`layer`, `cpu_capacity_millicores`, `memory_capacity_bytes`, `utilization_now.*`), trimmed to the grounding fields to keep the prompt short, including a node with null utilisation like the real `edge-node`.

In [5]:
rows = []
for k, e in F.EXAMPLES.items():
    g = canon_list(e["slos"])
    rows.append({"example": k, "used in": "fs2 only" if k in ("M1", "M2", "M3") else "fs1 and fs2", "signature": sig_of(e["slos"]),
                 "#SLOs": len(g), "kinds shown": "; ".join(kind_of(t) for t in g), "expression": e["expression"]})
exdf = pd.DataFrame(rows).set_index("example")
display(exdf)

print("\nLookup traps (restricted-to-layer answer vs the unrestricted one a careless model would give):")
for k in ("M4", "M5", "M6"):
    e = F.EXAMPLES[k]; layer = next(s for s in e["slos"] if s["type"] == "placement_location")["target"]["value"]
    nulls = [n["name"] for n in e["snapshot"]["nodes"] if n["utilization_now"]["cpu_util_percent"] is None]
    for s in e["slos"]:
        if s["target"]["op"] in ("highest", "lowest"):
            m, op = s["metric"], s["target"]["op"]
            print(f"  {k} {m} {op} on {layer}: demonstrated {s['target']['value']}  |  unrestricted would be {F.implicit(e['snapshot'], m, op)['target']['value']}  |  null-reading nodes skipped: {nulls}")

,used in,signature,#SLOs,kinds shown,expression
example,,,,,
M1,fs2 only,EEP,3,placement=cloud; cpu_capacity >= (explicit); memory_capacity >= (explicit),Please get svc/example-billing running in the cloud; the target node must have at least 4 CPU cores and at least 8 GB of memory.
M2,fs2 only,E,1,storage_capacity >= (explicit),"For svc/example-search, the target node needs at least 100 GB of storage capacity."
M3,fs2 only,EP,2,placement=edge; cpu_utilization <= (explicit),Can you put svc/example-gateway on edge nodes and keep CPU utilization below 45%?
M4,fs1 and fs2,EIP,3,placement=edge; storage_utilization <= (explicit); cpu_utilization lowest,"I would like svc/example-worker on the edge layer, with storage utilization below 65%, on the node with the lowest current CPU utilization."
M5,fs1 and fs2,IIP,3,placement=cloud; memory_capacity highest; port_utilization lowest,"Schedule svc/example-queue on the cloud side, on the node with the highest memory capacity and the lowest port utilization."
M6,fs1 and fs2,EIIP,4,placement=edge; memory_utilization <= (explicit); storage_utilization lowest; storage_capacity highest,"Go ahead and run svc/example-cache on the edge side, keep memory utilization below 60%, choose the node with the lowest current storage ..."



Lookup traps (restricted-to-layer answer vs the unrestricted one a careless model would give):
  M4 cpu_utilization lowest on edge: demonstrated 4.1  |  unrestricted would be 1.2  |  null-reading nodes skipped: ['edge-c']
  M5 memory_capacity highest on cloud: demonstrated 4125683712  |  unrestricted would be 8181379072  |  null-reading nodes skipped: []
  M5 port_utilization lowest on cloud: demonstrated 0.047  |  unrestricted would be 0.011  |  null-reading nodes skipped: []
  M6 storage_utilization lowest on edge: demonstrated 23.0  |  unrestricted would be 11.0  |  null-reading nodes skipped: ['edge-c']
  M6 storage_capacity highest on edge: demonstrated 62220779520  |  unrestricted would be 62220779520  |  null-reading nodes skipped: ['edge-c']


**Finding.** The six examples have the signatures E, EP, EEP, EIP, IIP and EIIP (16 SLOs in total). Every lookup example contains a trap that is only avoided by restricting to the requested layer and skipping null nodes: in M4 a cloud node has CPU utilisation 1.2 but the correct edge answer is 4.1 (and one edge node is null); in M5 an edge node has both the largest memory (8,181,379,072 against the correct cloud 4,125,683,712) and the lowest port utilisation (0.011 against 0.047); in M6 a cloud node has storage utilisation 11.0 but the correct edge answer is 23.0 (and one edge node has null readings). The one exception is M6's `storage_capacity highest`, whose unrestricted answer equals the demonstrated one; it only tests null skipping. All demonstrated values are computed from the example snapshots by code and checked by unit tests.

### 3a. Set-up 1: the three examples added to the paper's three (Examples 4-6 of `fs1`, also Examples 4-6 of `fs2`)

These replace the three undisclosed examples. The paper's visible examples are all single explicit thresholds, so the added ones are chosen to cover what is missing: **placement, layer-restricted highest/lowest lookups, several SLOs in one request.** All three are unit-neutral (no conversion), so they never contradict the paper's own prompt.

| # | Example | Why this example |
|---|---|---|
| **4 (M4)** | *"I would like svc/example-worker on the edge layer, with storage utilization below 65%, on the node with the lowest current CPU utilization."* Signature **EIP** (placement edge + explicit threshold + one lookup). | EIP is the single most common record signature (90 records, 17%). It teaches the full pattern in one example: placement, a threshold with `op: null` (the 4B's weak spot), and `cpu_utilization lowest` with `cmp "<="` (lowest is 58% of lookups; CPU is a top-3 lookup metric). **Trap:** a cloud node has a lower CPU utilisation (1.2) than any edge node, and one edge node has null readings, so the answer 4.1 requires restricting to the edge layer and skipping the null node. |
| **5 (M5)** | *"Schedule svc/example-queue on the cloud side, on the node with the highest memory capacity and the lowest port utilization."* Signature **IIP**, layer = cloud. | Covers the two weakest metrics in our runs: `memory_capacity highest` (the most frequent lookup, 53 SLOs, 24%) and `port_utilization lowest` (35 SLOs; F1 was 0.06 for the 9B). It shows **both directions** with the right `cmp` (`>=` for highest, `<=` for lowest) and the second layer (cloud). **Trap:** an edge node has both the largest memory and the lowest port utilisation, so ignoring the layer gives two wrong values. Capacities are copied in bytes from the snapshot (no conversion). |
| **6 (M6)** | *"Go ahead and run svc/example-cache on the edge side, keep memory utilization below 60%, choose the node with the lowest current storage utilization, and the node with the highest storage capacity."* Signature **EIIP** (4 SLOs). | The L4-style multi-constraint request (SLOs per record go up to 4): shows that **several lookups in one request are each resolved separately**, and covers `storage_utilization lowest` (47 SLOs, the second most frequent lookup metric) and `storage_capacity highest` (23). The explicit threshold here is `memory_utilization`, so all three utilisation thresholds appear across examples. **Trap:** a cloud node has lower storage utilisation (11.0), and one edge node has null utilisation *and* null storage total. The two lookups are answered by different nodes. |

### 3b. Set-up 2: the three new examples that replace the paper's three (Examples 1-3 of `fs2`)

These carry the **unit conventions** (the paper's three examples teach decimal GB, a MiB unit that never occurs in the data, and a flat snapshot), and the two most common simple signatures.

| # | Example | Why this example |
|---|---|---|
| **1 (M1)** | *"Please get svc/example-billing running in the cloud; the target node must have at least 4 CPU cores and at least 8 GB of memory."* Signature **EEP**. | EEP is the 4th most common signature (75 records). It demonstrates the two conversions the paper's prompt got wrong for this dataset in one go: **4 CPU cores -> 4000 millicores** (74 requests in the data, all phrased with "CPU cores") and **8 GB -> 8,589,934,592 bytes, binary** (gold uses 2^30 in 154 of 158 GB requests). Also the second placement value (cloud). |
| **2 (M2)** | *"For svc/example-search, the target node needs at least 100 GB of storage capacity."* Signature **E** (single SLO, no placement). | Single explicit SLO is the 2nd most common signature (82 records, 16%) and the simplest case; it shows that **no placement SLO is emitted when none is stated**, and repeats the binary-GB rule for the *other* capacity metric (storage; 83 requests) so the lesson is not tied to memory. The snapshot is present but irrelevant, as in most real requests. |
| **3 (M3)** | *"Can you put svc/example-gateway on edge nodes and keep CPU utilization below 45%?"* Signature **EP**. | EP is the 3rd most common signature (80 records). It shows the most frequent explicit kind, a **utilisation threshold with `cmp "<="` and `op: null`** (326 SLOs, the largest explicit group; this is the pattern the 4B turns into a spurious `highest`), on the edge layer. Together with Examples 4 and 6, all three utilisation thresholds (CPU, storage, memory) are demonstrated. |

## 4. Are the six examples representative of the dataset?

In [6]:
def example_sets():
    return {
        "paper's 3 visible": [paper_slos[k] for k in ("P1", "P2", "P3")],
        "fs1 (paper 3 + M4-M6)": [paper_slos[k] for k in ("P1", "P2", "P3")] + [ex_slos[k] for k in ("M4", "M5", "M6")],
        "fs2 (M1-M6)": [ex_slos[k] for k in ("M1", "M2", "M3", "M4", "M5", "M6")],
    }
ds_keys = Counter(key_of(t) for g in gold for t in g)
ds_kinds = Counter(kind_of(t) for g in gold for t in g)
ds_sigs = Counter(sig_of(r["label"]["slos"]) for r in recs)
rows = {}
for name, sets_ in example_sets().items():
    tl = [t for s in sets_ for t in canon_list(s)]
    sigs = {sig_of(s) for s in sets_}
    cov_keys = {key_of(t) for t in tl}; cov_kinds = {kind_of(t) for t in tl}
    imp_metrics = {t[1] for t in tl if t[3] in ("highest", "lowest")}
    all_imp = Counter(t[1] for g in gold for t in g if t[3] in ("highest", "lowest"))
    rows[name] = {
        "examples": len(sets_), "SLOs shown": len(tl),
        "placement share of SLOs shown": sum(t[0] == "placement_location" for t in tl) / len(tl),
        "implicit share of SLOs shown": sum(t[3] in ("highest", "lowest") for t in tl) / len(tl),
        "gold SLOs whose (type, metric, cmp, ext) is shown": sum(v for k, v in ds_keys.items() if k in cov_keys) / tot,
        "implicit SLOs whose (metric, direction) is shown": sum(1 for g in gold for t in g if t[3] in ("highest", "lowest") and kind_of(t) in cov_kinds) / sum(1 for g in gold for t in g if t[3] in ("highest", "lowest")),
        "records whose signature is shown": sum(v for k, v in ds_sigs.items() if k in sigs) / len(recs),
        "implicit SLOs whose metric is shown": sum(v for m, v in all_imp.items() if m in imp_metrics) / sum(all_imp.values()),
        "highest/lowest lookups shown": sum(1 for t in tl if t[3] in ("highest", "lowest")) if name != "paper's 3 visible" else 0,
        "cores->millicores shown": any(t[1] == "cpu_capacity" for t in tl),
        "binary GB shown": any(t[1] in ("memory_capacity", "storage_capacity") and t[4] in (8 * 2**30, 100 * 2**30) for t in tl),
    }
cov = pd.DataFrame(rows)
ds_ref = {"placement share of SLOs shown": share(lambda t: t[0] == "placement_location"), "implicit share of SLOs shown": share(lambda t: t[3] in ("highest", "lowest"))}
cov.insert(0, "dataset (reference)", [None, None, ds_ref["placement share of SLOs shown"], ds_ref["implicit share of SLOs shown"], None, None, None, None, None, None, None])
display(cov.T if False else cov)

shown = {kind_of(t) for s in example_sets()["fs2 (M1-M6)"] for t in canon_list(s)}
miss = pd.Series({k: v for k, v in ds_kinds.items() if k not in shown}).sort_values(ascending=False)
print("\nDataset SLO kinds NOT shown by the six examples (count in gold):"); display(miss.to_frame("gold SLOs").assign(share=lambda d: d["gold SLOs"] / tot).round(3))

,dataset (reference),paper's 3 visible,fs1 (paper 3 + M4-M6),fs2 (M1-M6)
examples,NaN,3,6,6
SLOs shown,NaN,3,13,16
placement share of SLOs shown,0.333618,0.0,0.230769,0.3125
implicit share of SLOs shown,0.189050,0.0,0.384615,0.3125
"gold SLOs whose (type, metric, cmp, ext) is shown",NaN,0.244654,0.857998,0.9213
"implicit SLOs whose (metric, direction) is shown",NaN,0.0,0.58371,0.58371
records whose signature is shown,NaN,0.15739,0.362764,0.660269
implicit SLOs whose metric is shown,NaN,0.0,0.891403,0.891403
highest/lowest lookups shown,NaN,0,5,5
cores->millicores shown,NaN,False,False,True



Dataset SLO kinds NOT shown by the six examples (count in gold):


,gold SLOs,share
storage_utilization highest,24,0.021
memory_capacity lowest,23,0.020
memory_utilization lowest,14,0.012
cpu_utilization highest,10,0.009
port_utilization highest,6,0.005
storage_capacity lowest,5,0.004
cpu_capacity lowest,5,0.004
memory_utilization highest,4,0.003
cpu_capacity highest,1,0.001


**Finding.** **Representativeness.** By SLO kind (type, metric, comparison, extremum tag), the paper's three visible examples cover 24.5% of the gold SLOs, set-up 1 (paper 3 + M4-M6) covers 85.8%, and set-up 2 (M1-M6) covers **92.1%**. By record signature the coverage is 15.7%, 36.3% and **66.0%**, and for implicit metrics set-up 1 and 2 both cover 89.1% of the lookups by metric and 58.4% by metric and direction (the paper's examples cover none). Set-up 2 matches the dataset's share of placement SLOs (31% shown against 33%) and deliberately over-weights lookups (31% shown against 19%) because they are the hard part. Not shown are 7.9% of the gold SLOs, all rare lookup directions (for example `storage_utilization highest`, `memory_capacity lowest`, `memory_utilization lowest`), where the model must generalise the highest/lowest rule from the other metrics; and the signatures P-only (10.6%), IP (7.9%), EE (6.9%) and EEIP (5.2%), which are combinations of elements that do appear. Set-up 1 is the less representative of the two because it keeps the paper's three single-threshold examples and has neither unit conversion shown.

In [7]:
# leakage / copying check: do the examples copy dataset expressions or clause templates?
ds_norm = {norm(r["input"]["intent"]["expression"], r["input"]["intent"]["service"]) for r in records}
def clauses(e): return [c.strip().rstrip(".?") for c in re.split(r",\s*(?:and\s+)?|;\s*|\s+and\s+(?=(?:keep|make|pick|place|run|schedule|the node))", e) if c.strip()]
ds_clauses = {c for r in records for c in clauses(norm(r["input"]["intent"]["expression"], r["input"]["intent"]["service"]))}
rows = []
for k, e in F.EXAMPLES.items():
    n = norm(e["expression"], e["service"])
    best = max(difflib.SequenceMatcher(None, n, d).ratio() for d in ds_norm)
    cl = clauses(n)
    rows.append({"example": k, "copies a dataset expression": n in ds_norm, "closest dataset expression (similarity)": round(best, 2),
                 "clauses": len(cl), "clauses that also occur in the dataset": sum(c in ds_clauses for c in cl)})
display(pd.DataFrame(rows).set_index("example"))

,copies a dataset expression,closest dataset expression (similarity),clauses,clauses that also occur in the dataset
example,,,,
M1,False,0.67,2,0
M2,False,0.84,2,1
M3,False,0.84,2,1
M4,False,0.80,3,0
M5,False,0.74,2,0
M6,False,0.76,5,2


**Finding.** None of the examples copies a dataset expression (after replacing numbers and service names), and the closest dataset expression has similarity 0.67-0.84. Examples M2, M3 and M6 share 1, 1 and 2 clauses with the dataset (typically the standard "keep X utilization below N%" clause), M1, M4 and M5 share none. So the examples use the dataset's vocabulary but not its sentence frames. Because the dataset has a closed grammar (525 templates), some gain from few-shot may still come from familiarity with that vocabulary; this should be kept in mind when interpreting the results.

## 5. Prompt set-up 1: `fs1_verbatim+3_prompt.txt`

**Composition:** the paper's zero-shot **system prompt, unchanged** (the same file used by all zero-shot runs) + the paper's **Examples 1-3 verbatim** (including their known problems: decimal GB, MiB, flat snapshot field names, the extra `"range": null` key) + **Examples 4-6 = M4, M5, M6** + the query block. This keeps the replication of the paper as faithful as the published information allows, and the only addition is what the paper does not disclose.

**Why keep the paper's known problems?** Set-up 1 is the reference point: it tells us how far the paper's few-shot design carries lightweight models. The effect of fixing the problems is then measured by set-up 2.

**What the added examples do and do not do in this set-up:** they teach placement, `op: null` thresholds, layer-restricted lookups and the `<=`/`>=` mapping; because they are unit-neutral they do not interact with the paper's `cpu_capacity=cores` line or with its decimal-GB Example 2. (Their snapshots use the real field names, while the paper's examples use flat ones; this mixture is a property of the paper's published examples and is kept.)

In [8]:
s1, u1 = load_fs_prompt(F.FS1_NAME)
zs_sys = (ROOT / "prompts/zs_cot_p_paper_system.txt").read_text()
print("fs1 system prompt identical to the zero-shot paper system prompt:", s1.strip() == zs_sys.strip())
print("examples in fs1 user template:", re.findall(r"^Example \d$", u1, re.M))
parts = re.split(r"^Example \d\n", u1, flags=re.M)[1:]
rows = [{"example #": i + 1, "source": ("paper (verbatim)" if i < 3 else "added: " + ["M4", "M5", "M6"][i - 3]),
         "chars": len(p), "has 'range' key": '"range"' in p, "units shown": ("decimal GB" if "2000000000" in p else "MiB" if "MiB" in p else "-")} for i, p in enumerate(parts)]
display(pd.DataFrame(rows).set_index("example #"))
print("system prompt chars:", len(s1), "| user template chars:", len(u1))

fs1 system prompt identical to the zero-shot paper system prompt: True
examples in fs1 user template: ['Example 1', 'Example 2', 'Example 3', 'Example 4', 'Example 5', 'Example 6']


,source,chars,has 'range' key,units shown
example #,,,,
1,paper (verbatim),477,True,-
2,paper (verbatim),485,True,decimal GB
3,paper (verbatim),507,True,MiB
4,added: M4,1824,False,-
5,added: M5,1562,False,-
6,added: M6,2167,False,-


system prompt chars: 1716 | user template chars: 7082


**Finding.** Set-up 1 is the paper's prompt unchanged (system prompt verified identical to the zero-shot paper system prompt) with its three visible examples verbatim (477-507 characters each, all with the `"range"` key, one with decimal GB, one with MiB) followed by M4, M5, M6 (1,562-2,167 characters each, longer because they carry real-format snapshots of 3-4 nodes). System prompt 1,716 characters, user template 7,082 characters. The paper's known problems are kept on purpose; set-up 2 measures the effect of fixing them.

## 6. Prompt set-up 2: `fs2_unitFixed+6_prompt.txt`

**Composition:** the paper's system prompt with the changes listed below + **six curated examples M1-M6** (the paper's three examples are removed) + the query block.

**The goal of every change is the same:** remove ambiguity that the gold labels resolve in one particular way, so that a model's mistakes are about extraction and grounding, not about guessing a convention. Wherever the paper's prompt leaves a choice open, the choice is made as the dataset made it.

In [9]:
rows = []
for i, (old, new, label_) in enumerate(F.FS2_CHANGES, 1):
    rows.append({"#": i, "change": label_, "before (paper)": old.strip().replace("\n", " / ")[:150], "after (fs2)": new.strip().replace("\n", " / ")[:150]})
display(pd.DataFrame(rows).set_index("#"))
print("unified diff of the system prompt (fs1 = paper  ->  fs2):\n")
for l in difflib.unified_diff(F.system_fs1().splitlines(), F.system_fs2().splitlines(), "fs1 (paper)", "fs2", lineterm="", n=0): print(l)

,change,before (paper),after (fs2)
#,,,
1,canonical cpu unit: cores -> millicores,cpu_capacity=cores;,cpu_capacity=millicores;
2,"explicit unit-conventions block (cores->millicores factor, binary data sizes, snapshot already canonical)",port_utilization=percent; / / Procedure:,"port_utilization=percent; / / Unit conventions: / - 1 core = 1000 millicores, so ""4 CPU cores"" means cpu_capacity 4000 (millicores). / ..."
3,removes the ambiguous 'SI/IEC' wording,using standard unit conventions (SI/IEC for data sizes; time and rate unit conversions as standard).,using the unit conventions above (time and rate unit conversions as standard).
4,procedure step that points to the snapshot rule,- Emit the JSON.,"- If the expression asks for an extremum, follow ""Snapshot use"" below before emitting. / - Emit the JSON."
5,precautionary snapshot-verification rule (replaces the paper's 'Snapshot use' block),Snapshot use: / - Only consult the snapshot when the expression requests an extremum (highest/lowest/etc.). / - When comparing snapshot ...,"Snapshot use (MANDATORY for highest/lowest requests): / - Explicit thresholds (""below 45%"", ""at least 4 CPU cores"") do not need the snap..."


unified diff of the system prompt (fs1 = paper  ->  fs2):

--- fs1 (paper)
+++ fs2
@@ -32 +32 @@
-cpu_capacity=cores;
+cpu_capacity=millicores;
@@ -38,0 +39,5 @@
+Unit conventions:
+- 1 core = 1000 millicores, so "4 CPU cores" means cpu_capacity 4000 (millicores).
+- Data sizes are binary: 1 KB = 1 KiB = 1024 bytes; 1 MB = 1 MiB = 1024^2 bytes; 1 GB = 1 GiB = 1024^3 = 1073741824 bytes; 1 TB = 1 TiB = 1024^4 bytes. So "8 GB" means 8589934592 bytes.
+- The snapshot already uses the canonical units (millicores, bytes, percent): copy its values unchanged.
+
@@ -42 +47,2 @@
-- Normalize numeric values into the canonical unit for that metric using standard unit conventions (SI/IEC for data sizes; time and rate unit conversions as standard).
+- Normalize numeric values into the canonical unit for that metric using the unit conventions above (time and rate unit conversions as standard).
+- If the expression asks for an extremum, follow "Snapshot use" below before emitting.
@@ -45,3 +51,6 @@
-S

**Finding.** Set-up 2 differs from the paper's system prompt in five places, all visible in the diff: the canonical cpu unit, a new unit-conventions block, the removal of the ambiguous "SI/IEC" wording, one procedure step pointing to the snapshot rule, and the replacement of the paper's three-line snapshot block by the mandatory verification rule. Everything else in the system prompt is byte-identical. The examples are replaced as described in sections 3a and 3b.

**Justification of each change in set-up 2**

| # | Change | Justification (evidence) |
|---|---|---|
| 1 | `cpu_capacity=cores` -> `millicores` | The gold labels store millicores (80 of 80 cpu_capacity SLOs); the intents say "N CPU cores". With the paper's `cores` line, a correct answer is scored as an error: moving to millicores took the 9B's cpu_capacity F1 from 0.00 to 0.94 in the earlier experiment. |
| 2 | New **Unit conventions** block: 1 core = 1000 millicores; **data sizes binary** (1 GB = 1024^3 bytes, KB/MB/TB likewise); the snapshot already uses canonical units, copy unchanged | The gold labels use binary GB in 154 of 158 GB requests, while the paper's prompt only says "SI/IEC" (which allows both) and its Example 2 teaches decimal. The 27B in thinking mode used decimal in 80 of 158 cases and lost about 0.07 F1. The "copy unchanged" line prevents unit re-conversion of snapshot values that are already canonical. |
| 3 | `standard unit conventions (SI/IEC for data sizes; ...)` -> `the unit conventions above (...)` | Removes the ambiguous wording that conflicts with change 2. |
| 4 | Procedure step: *if the expression asks for an extremum, follow "Snapshot use" before emitting* | Puts the snapshot check into the model's step-by-step procedure, so it is not just a trailing rule. |
| 5 | **Precautionary snapshot rule** replaces the paper's "Snapshot use" block: MANDATORY; you MUST verify in the snapshot; never leave `target.value` null and never guess; candidate nodes = requested layer (else all), ignore null nodes; copy the extreme node's value unchanged; set `op` and `cmp` (>= highest, <= lowest) | **The researcher's hypothesis:** an explicit, forceful verification rule makes the model read the snapshot. Baseline for the hypothesis (existing zero-shot runs): the right operator but `value: null` for 59 of 221 lookups (4B) and 109 of 221 (9B), and 221 of 221 for the 27B. The paper's wording ("Only consult the snapshot when...") is permissive and never says the value must be filled in. **Beyond the minimal ask (and separable as an ablation):** the layer-scope sentence, the null-skipping sentence and the `cmp` mapping; they are there because wrong-layer answers, null nodes and wrong operators were the other observed lookup errors, and they match what Examples 4-6 demonstrate. |
| 6 | Examples: the paper's three removed, M1-M6 used | The paper's examples teach decimal GB (Example 2), a MiB unit absent from the data (Example 3), a snapshot schema that does not exist (all three), and an unused `"range"` key; none shows placement, lookups or several SLOs. See section 3b. |

**Deliberately not changed:** the JSON schema and the "Output MUST be JSON only" line; `bandwidth=Mbps` (the paper's Table 2 says Bps, but the dataset has **no** bandwidth SLO, so this inconsistency cannot affect scores); the placement rule; the format of the user message.

**How the hypothesis can be tested later (not run yet):** compare, on the 221 gold highest/lowest SLOs, the counts of "right operator but `value: null`", "right number" and "wrong operator" between zero-shot, fs1 and fs2, per model. The same table was used in the earlier notebooks.

In [10]:
# baseline for the set-up-2 hypothesis: how often do the existing zero-shot runs answer a lookup with value null?
def latest(pattern):
    return max(d for d in (ROOT / "outputs/runs").glob(pattern) if (d / "predictions.jsonl").exists() and "smoke" not in d.name)
base = {}
for name, pat in {"4B zero-shot (p_mc)": "zs_p_mc_qwen35-4b_2*", "9B zero-shot (p_mc)": "zs_p_mc_qwen35-9b_2*"}.items():
    base[name] = A.extremum_outcomes(recs, load_predictions(latest(pat) / "predictions.jsonl"))
b = pd.DataFrame(base).fillna(0).astype(int)
b.loc["share with value null"] = (b.loc["right op, value null (snapshot not read)"] / 221).round(3)
display(b)

,4B zero-shot (p_mc),9B zero-shot (p_mc)
metric not predicted,18.000,8.000
"right op, correct number",88.000,35.000
"right op, value null (snapshot not read)",59.000,109.000
"right op, wrong number",48.000,20.000
wrong op (e.g. threshold / opposite extreme),8.000,49.000
share with value null,0.267,0.493


**Finding.** Baseline for the set-up 2 hypothesis (existing zero-shot runs, millicores prompt): on the 221 gold highest/lowest SLOs the 4B writes the right operator but `value: null` in 59 cases (26.7%) and the correct number in 88 (39.8%); the 9B writes `null` in 109 cases (49.3%), the correct number in 35 (15.8%) and a wrong operator in 49 (the 27B wrote `null` in all 221). If the hypothesis holds, the "value null" share should fall and the correct-number share should rise under set-up 2, and the wrong-operator count should fall because of the explicit `>=`/`<=` mapping. The 4B's remaining errors (spurious `highest` on thresholds) are expected to be addressed by the threshold examples rather than by the rule.

## 7. Validation of the two prompt files

In [11]:
checks = {}
for name in (F.FS1_NAME, F.FS2_NAME):
    sysm, user = load_fs_prompt(name)
    on_disk = (ROOT / "prompts" / name).read_text()
    rec = next(r for r in recs if r["index"] == 25)
    msgs = build_messages_fs(rec, snapshot_compact(snapshot), name)
    checks[name] = {
        "file equals generator output": on_disk == F.file_text(name),
        "6 numbered examples": len(re.findall(r"^Example \d$", user, re.M)) == 6,
        "each placeholder once": all(user.count(p) == 1 for p in ("{service}", "{expression}", "{snapshot}")),
        "renders a real record without leftover placeholders": not any(p in msgs[1]["content"] for p in ("{service}", "{expression}", "{snapshot}")),
        "ends with 'Answer:'": msgs[1]["content"].rstrip().endswith("Answer:"),
        "all example outputs parse as JSON": all(parse_response(b.split("Output:\n", 1)[1])["parse_ok"] for b in re.split(r"^Example \d\n", user, flags=re.M)[1:]),
    }
display(pd.DataFrame(checks))
ok = {k: all(v.values()) for k, v in checks.items()}
print("all structural checks pass:", ok)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_fewshot.py"], cwd=ROOT, capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1])

,fs1_verbatim+3_prompt.txt,fs2_unitFixed+6_prompt.txt
file equals generator output,True,True
6 numbered examples,True,True
each placeholder once,True,True
renders a real record without leftover placeholders,True,True
ends with 'Answer:',True,True
all example outputs parse as JSON,True,True


all structural checks pass: {'fs1_verbatim+3_prompt.txt': True, 'fs2_unitFixed+6_prompt.txt': True}


9 passed in 0.16s


**Finding.** Both prompt files pass all structural checks: they equal the generator output, contain six numbered examples, one occurrence of each placeholder, render a real record without leftover placeholders, end with `Answer:`, and every example output parses as JSON. The nine unit tests for the few-shot module pass (the suite has 23 tests in total).

In [12]:
# prompt length for a real record (Qwen3.5 chat template) and the context window the runs will need
VENV_PY = ROOT / ".venv-vllm/bin/python"
rec = next(r for r in recs if r["index"] == 25)
snap_str = snapshot_compact(snapshot)
variants = {"zero-shot, paper prompt (p_paper)": build_messages(rec, snap_str, regime="zs"),
            "zero-shot, millicores prompt (p_mc)": [{"role": "system", "content": (ROOT / "prompts/zs_cot_p_mc_system.txt").read_text()}, build_messages(rec, snap_str, regime="zs")[1]],
            "fs1_verbatim+3": build_messages_fs(rec, snap_str, F.FS1_NAME), "fs2_unitFixed+6": build_messages_fs(rec, snap_str, F.FS2_NAME)}
out = subprocess.run([str(VENV_PY), "scripts/count_tokens.py", "Qwen/Qwen3.5-4B"], input=json.dumps(list(variants.values())), cwd=ROOT, capture_output=True, text=True)
counts = json.loads(out.stdout.strip().splitlines()[-1])
tk = pd.DataFrame({"prompt tokens": counts}, index=list(variants))
tk["added vs zero-shot (p_mc)"] = tk["prompt tokens"] - tk.loc["zero-shot, millicores prompt (p_mc)", "prompt tokens"]
snap_tokens = counts[0] - 0
for budget, label_ in ((2048, "zs: + 2,048 answer tokens"), (4096 + 1024, "zs-think: + 4,096 thinking + 1,024 answer")):
    tk[label_] = tk["prompt tokens"] + budget
std = lambda n: next(w for w in (8192, 12288, 16384, 24576) if n <= w)
tk["max_model_len needed (zs)"] = tk["zs: + 2,048 answer tokens"].map(std)
tk["max_model_len needed (zs-think)"] = tk["zs-think: + 4,096 thinking + 1,024 answer"].map(std)
display(tk)
print("For the earlier runs max_model_len was 8,192 (zs) and 12,288 (zs-think).")

,prompt tokens,added vs zero-shot (p_mc),"zs: + 2,048 answer tokens","zs-think: + 4,096 thinking + 1,024 answer",max_model_len needed (zs),max_model_len needed (zs-think)
"zero-shot, paper prompt (p_paper)",5538,-2,7586,10658,8192,12288
"zero-shot, millicores prompt (p_mc)",5540,0,7588,10660,8192,12288
fs1_verbatim+3,7936,2396,9984,13056,12288,16384
fs2_unitFixed+6,8887,3347,10935,14007,12288,16384


For the earlier runs max_model_len was 8,192 (zs) and 12,288 (zs-think).


**Finding.** For record 25, the zero-shot prompt is 5,540 tokens; set-up 1 is 7,936 (+2,396) and set-up 2 is 8,887 (+3,347), 43% and 60% longer. With the context limits used so far (8,192 for zero-shot, 12,288 for thinking) neither few-shot prompt fits: zero-shot runs need 12,288 and thinking runs 16,384. The system prompt, the examples and the single snapshot form a constant prefix shared by all records, so prefix caching should absorb most of the extra prefill cost, but the longer context and KV memory must be provisioned, and energy per record will be somewhat higher.

## 8. Open items and next steps

- **Experiment policy not yet defined** (by the researcher): which set-ups to run (zero-shot and/or thinking), for 4B and 9B, in which order; whether to add controls (e.g. paper's 3 examples only, random examples, example-count ablation, chat-turn format).
- **The runner does not yet take these prompt files.** `scripts/run_zs_cot.py` still loads the zero-shot prompts; a `--prompt-file` option (using `src.prompts.load_fs_prompt`) is a small change that I left until the policy is defined. Few-shot prompts are longer, so the context window must be raised (see the token table above).
- **Predictions to be checked once runs exist** (written before running): fs1 should raise the 4B above its zero-shot level mainly by removing the spurious `highest` tag on thresholds; fs1 may *lower* memory/storage capacity scores because the paper's Example 2 teaches decimal GB; fs2 should restore them and, if the hypothesis holds, cut the share of lookups answered with `value: null`.
- **Caveats to carry into the report:** closed grammar (examples written in the dataset's vocabulary can inflate gains through template familiarity); one greedy run per cell and no confidence intervals; extremum convention follows the requested layer (assumption 1); three clarifications in the fs2 snapshot rule go beyond the minimal ask.

## Appendix: the two prompt files (system prompt, then user template with placeholders)

In [13]:
for name in (F.FS1_NAME,):
    print("#" * 30, name, "#" * 30); print((ROOT / "prompts" / name).read_text())

############################## fs1_verbatim+3_prompt.txt ##############################
=== SYSTEM ===
Instruction:
You are an information extraction system that converts natural-language service placement intents into SLO labels.

IMPORTANT:
Output MUST be JSON only.

Return ONLY valid JSON matching:
{
 "service": "<service>",
 "slos": [
  {
   "metric": <string or null>,
   "target": {
    "cmp": "<=" | ">=" | "between" | "=" | null,
    "op": "highest" | "lowest" | "set" | null,
    "value": <number | string | null>
   },
   "type": "placement_location" | "static_compute_resource" | "dynamic_compute_resource" | "node_network_resource",
   "unit": <string or null>
  }
 ]
}

Constraints:
1) Use only these types/metrics:
- static_compute_resource: cpu_capacity, memory_capacity, storage_capacity
- dynamic_compute_resource: cpu_utilization, memory_utilization, storage_utilization
- node_network_resource: bandwidth, port_utilization
- placement_location: metric=null, unit=null

2) Use can

In [14]:
for name in (F.FS2_NAME,):
    print("#" * 30, name, "#" * 30); print((ROOT / "prompts" / name).read_text())

############################## fs2_unitFixed+6_prompt.txt ##############################
=== SYSTEM ===
Instruction:
You are an information extraction system that converts natural-language service placement intents into SLO labels.

IMPORTANT:
Output MUST be JSON only.

Return ONLY valid JSON matching:
{
 "service": "<service>",
 "slos": [
  {
   "metric": <string or null>,
   "target": {
    "cmp": "<=" | ">=" | "between" | "=" | null,
    "op": "highest" | "lowest" | "set" | null,
    "value": <number | string | null>
   },
   "type": "placement_location" | "static_compute_resource" | "dynamic_compute_resource" | "node_network_resource",
   "unit": <string or null>
  }
 ]
}

Constraints:
1) Use only these types/metrics:
- static_compute_resource: cpu_capacity, memory_capacity, storage_capacity
- dynamic_compute_resource: cpu_utilization, memory_utilization, storage_utilization
- node_network_resource: bandwidth, port_utilization
- placement_location: metric=null, unit=null

2) Use ca